# Look-ahead · Shared Base  `[EVAL]`

**The GRPO look-ahead contrast against ONE Base.** Both GRPO runs start from the same untrained
Llama-3.2-1B, and each drew its own 96 iteration-0 conversations. Every other `lookahead` family keeps
the two draws apart (iteration 0 = a free noise floor). The GRPO paper reports one Base instead
(Lior, 2026-09-24): the two draws pooled — 192 conversations, two per persona (both runs share the
iteration-0 persona order). This family recomputes every Base-dependent number the paper quotes on
that single Base, under both graders:

1. **Levels, the K contrast, the complete score table, gains over the Base** — K contrasts start at
   iteration 1, so the Holm family is iterations 1..10 within (judge, metric).
2. **The process coder (MIPROC)** — code mix, yields, responsiveness, the within-session trajectory,
   parity with MITI/PCT (21 states), and the **change-talk persistence** split: P(patient change talk
   | the patient's previous code), pooled and per conversation, with its persona-paired K contrast.
3. **Sentence-embedding space** — drift from the shared Base centroid, the cosine between the two
   runs' displacements, template similarity and the between-conversation variance share.
4. **The judge-free over-praise marker, session length, turn length.**
5. **Saturation (Appendix E of the paper)** — per-conversation SD + ceiling shares by iteration, the SD
   trend, per-state cross-judge agreement over the 21 states, sign preservation over all pairs of the 21
   states, and the judges' level offset.

Backing module: `eda_analysis/shared_base.py` (+ `process.persistence_by_state` /
`persistence_metrics` / `conditioned_yield`). GRPO arms only (the paper's data).

In [ ]:
import sys, os, time
_p = os.path.abspath(".")                      # find eda/ (the dir holding eda_analysis/) from any depth
while _p != os.path.dirname(_p) and not os.path.isdir(os.path.join(_p, "eda_analysis")):
    _p = os.path.dirname(_p)
sys.path.insert(0, _p)
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt
import matplotlib.transforms as mtransforms
pd.set_option("display.width", 185, "display.max_columns", 60, "display.max_colwidth", 120)

import eda_analysis
from eda_analysis import exports, plotting, behavior, reliability, process, text
from eda_analysis import shared_base
from eda_analysis.constants import BOOT_SEED, DISPLAY_NAMES, set_active_judge, judge_dirname, PRIMARY_JUDGE_TAG, arm_label
from eda_analysis.plotting.lookahead import K_STYLE

cfg = eda_analysis.EdaConfig(family="lookahead/shared_base", methods=["GRPO"],
                             judge=os.environ.get("EDA_JUDGE", ""))
S = eda_analysis.notebook_setup(cfg)
exports.reset_results()
exports.save_provenance(cfg, S.SCORES)
_T0 = time.time()

## 1 · Levels, the K contrast, the complete score table, gains

`per_arm` frames give both runs the pooled Base as iteration 0 (identical rows); `single` frames attach
it to the K=0 run only (the 21-state view used in §5).

In [ ]:
KA = S.ARMS
G_ARMS = [a.label for a in KA]
assert G_ARMS == ["GRPO_LA0", "GRPO_LA5"], G_ARMS
PAL = plotting.arm_palette(G_ARMS)
SC = eda_analysis.scores_by_judge(S)                 # {judge: scores_long}, primary FIRST
JUDGES = list(SC)
PRIMARY, HELDOUT = JUDGES[0], (JUDGES[1] if len(JUDGES) > 1 else None)
SB = shared_base.share_base_frames(SC, mode="per_arm")
ONE = shared_base.share_base_frames(SC, mode="single")
_b = SB[PRIMARY][SB[PRIMARY].iteration == 0]
print(f"judges {JUDGES} | shared Base: {_b.groupby('arm').file_index.nunique().to_dict()} conversations per run "
      f"({_b.base_draw.nunique()} draws)")

LV = shared_base.levels(SB)
KC = shared_base.k_contrast(SB)
SIG = shared_base.significant_iterations(KC)
GAINS = shared_base.gains(SB, metrics=tuple(m for m in ("Q1Q2", "Q1", "Q2", "WAI-SR", "CSQ-8", "MI-SAT", "MITI", "PCT", "MICI")))
display(SIG); display(GAINS[GAINS.metric == "Q1Q2"].round(3))

exports.save_table(LV.round(4), "levels_long", caption=(
    "Per (judge, run, rubric, iteration): n, mean, sd (ddof=1) and SE over the state's conversations, on ONE "
    "shared Base: iteration 0 is the two GRPO runs' base draws pooled (192 conversations, identical rows under "
    "both runs). Never average the two graders."))
exports.save_table(KC.round(4), "k_contrast", caption=(
    "Persona-paired K contrast at iterations 1..10 per (judge, rubric): the EDA sign (mean_delta, dz = K=0 − K=5) "
    "and the paper's (delta/dz/CI _K5_minus_K0); better = the arm the contrast favours, read through "
    "lower-is-better (MICI). Holm across ITERATIONS 1..10 within (judge, rubric) — iteration 0 is one policy "
    "with one Base, so it is not a test."))
exports.save_table(SIG, "significant_iterations", caption=(
    "Per (judge, rubric): the iterations where the K contrast clears Holm, split by the arm it favours "
    "(k_contrast). first_K5_lead = the first iteration whose point estimate favours K=5."))
exports.save_table(GAINS.round(4), "gains", caption=(
    "Each run's gain over the shared Base, persona-paired (the Base's persona value = the mean of its two "
    "conversations): anchor 'last' = both runs at iteration 10; 'best_K0' = the K=0 run at its best trained "
    "iteration on Q1+Q2 under THAT judge, the K=5 run still at iteration 10. ratio_K5_over_K0 (on the K=5 "
    "rows) = K=5's gain / K=0's gain at that anchor."))
ST = {}
for j in JUDGES:
    ST[j] = shared_base.score_table(LV, KC, j)
    exports.save_table(ST[j], f"score_table_{j}", caption=(
        f"[{j}] THE COMPLETE SCORE TABLE: every instrument's mean for every model — the shared Base, then the "
        "K=0 run and the K=5 run at iterations 1..10. '*' marks the better run's value at an iteration where the "
        "persona-paired K contrast clears Holm across iterations 1..10 (k_contrast; lower-is-better read for "
        "MICI). Levels are in this grader's own units."))
display(ST[PRIMARY])

In [ ]:
# The full battery in levels on the shared Base, one 3x3 grid per grader (the paper redraws it at page
# width from levels_long + k_contrast in two rows; this is the family's own record of the same cells).
_METRICS = [m for m in ("Q1Q2", "Q1", "Q2", "WAI-SR", "CSQ-8", "MI-SAT", "MITI", "PCT", "MICI") if m in set(LV.metric)]
for j in JUDGES:
    fig, axg = plt.subplots(3, 3, figsize=(11.6, 8.4), sharex=True)
    for ax, m in zip(axg.ravel(), _METRICS):
        lv = LV[(LV.judge == j) & (LV.metric == m)]
        kc = KC[(KC.judge == j) & (KC.metric == m)]
        for K in (0, 5):
            arm, st = f"GRPO_LA{K}", K_STYLE[K]
            dm = lv[lv.arm == arm].sort_values("iteration")
            ax.fill_between(dm.iteration, dm["mean"] - dm.se, dm["mean"] + dm.se, color=PAL[arm], alpha=0.15, lw=0)
            ax.plot(dm.iteration, dm["mean"], ls=st["ls"], marker=st["marker"], ms=3.5, lw=1.4, color=PAL[arm],
                    label=arm_label(arm))
        _base = lv[(lv.iteration == 0)]["mean"]
        if len(_base):
            ax.axhline(float(_base.iloc[0]), ls=":", lw=0.8, color="#555555")
        _tr = mtransforms.blended_transform_factory(ax.transData, ax.transAxes)
        for it in kc[kc.p_holm < 0.05].iteration:
            ax.text(int(it), 0.98, "*", transform=_tr, ha="center", va="top", fontsize=10, color="#333333")
        ax.set_title(DISPLAY_NAMES.get(m, m) + ("  (lower = better)" if m == "MICI" else ""), fontsize=9)
        ax.grid(True, alpha=0.3); ax.tick_params(labelsize=7.5)
        ax.set_xticks(range(0, 11, 2))
    for ax in axg[-1]:
        ax.set_xlabel("iteration (0 = Base)", fontsize=8)
    axg[0, 0].legend(fontsize=7.5, frameon=False, loc="best")
    fig.suptitle(f"GRPO, every instrument on ONE shared Base (dotted) — mean ± SE — grader: {j}   "
                 "('*' = K contrast clears Holm across iterations 1..10)", fontsize=10)
    fig.tight_layout()
    exports.save_fig(fig, f"levels_grid_{j}", caption=(
        f"[{j}] Every instrument by iteration for the two GRPO runs from ONE shared Base (iteration 0, dotted line: "
        "the two base draws pooled, 192 conversations), mean ± SE. '*' = the persona-paired K contrast clears Holm "
        "across iterations 1..10 at that iteration (k_contrast). Source: levels_long."))
    plt.show()

## 2 · The process coder on the shared Base, and change-talk persistence

In [ ]:
TAGS = {judge_dirname(PRIMARY_JUDGE_TAG): ""}
for t in reliability.second_judge_tags():
    TAGS[judge_dirname(t)] = t
PROC = {}
try:
    for j, tag in TAGS.items():
        set_active_judge(tag, 0)
        conv = process.load_miproc(KA)
        if conv.empty:
            print(f"{j}: no MIPROC scores — skipped"); continue
        miti = behavior.load_miti_behavior(KA, attach_persona=False)
        pct = behavior.load_pct_behavior(KA, attach_persona=False)
        PROC[j] = shared_base.process_tables(conv, miti, pct, KA, judge=j)
finally:
    set_active_judge("", 0)
PJ = list(PROC)
KP = pd.concat([PROC[j]["k_paired"] for j in PJ], ignore_index=True)
KPE = pd.concat([PROC[j]["k_persistence"] for j in PJ], ignore_index=True)
for j in PJ:
    t = PROC[j]
    exports.save_table(t["levels"].round(4), f"process_levels_{j}", caption=(
        f"[{j}] Per (run, iteration): mean and SE over the state's conversations of every per-conversation process "
        "metric (process.PROCESS_METRIC_LABELS), on the shared Base (iteration 0 = 192 conversations). Opener excluded."))
    exports.save_table(t["yield"].round(4), f"yield_{j}", caption=(
        f"[{j}] Per (run, iteration, therapist code): n turns and the share of the patient's next utterance coded "
        "CT / ST / NEU (Wilson 95% on CT), shared Base."))
    exports.save_table(t["responsiveness"].round(4), f"responsiveness_{j}", caption=(
        f"[{j}] Per (run, iteration, preceding patient code): the distribution of the therapist's next code, shared Base."))
    exports.save_table(t["ct_trajectory"].round(4), f"ct_trajectory_{j}", caption=(
        f"[{j}] Per (run, iteration, patient turn bin): change-talk / sustain-talk share of patient utterances, n, and "
        "the share of the state's conversations reaching the bin; shared Base."))
    exports.save_table(t["persistence"].round(4), f"persistence_{j}", caption=(
        f"[{j}] CHANGE-TALK PERSISTENCE, pooled over turns: per (run, iteration, previous patient code) the "
        "distribution of the patient's NEXT code. prev_code = CT: p_ct = change talk persists, p_st = relapse to "
        "sustain talk; prev_code = ST: p_ct = conversion out of sustain talk. Wilson 95% on p_ct. Shared Base."))
    exports.save_table(t["persistence_levels"].round(4), f"persist_levels_{j}", caption=(
        f"[{j}] Per (run, iteration): mean / SE / count over conversations of the per-conversation persistence "
        "metrics (process.PERSISTENCE_LABELS) — the unit of the persona-paired contrast in k_persistence."))
    exports.save_table(t["conditioned_yield"].round(4), f"cond_yield_{j}", caption=(
        f"[{j}] The yield split by what the patient said BEFORE the therapist's turn: per (run, iteration, previous "
        "patient code, therapist code) n and P(next patient = CT), Wilson 95%; th_code ALL = every therapist code. "
        "A code whose yield after change talk matches ALL is riding the patient's momentum."))
    if "parity_pooled" in t:
        exports.save_table(t["parity"].round(4), f"parity_{j}", caption=(
            f"[{j}] MIPROC counts vs the same grader's MITI / PCT counts per state (21 states: ONE Base)."))
        exports.save_table(t["parity_pooled"].round(4), f"parity_pooled_{j}", caption=(
            f"[{j}] Fisher-z pooled within-state ρ and mean |Δ| per count over the 21 states (one Base)."))
exports.save_table(KP.round(4), "k_process_paired", caption=(
    "Persona-paired K=0 − K=5 on every per-conversation process metric, per grader, iterations 1..10 (Holm across "
    "those iterations within (judge, metric)); better = the arm favoured, read through process.LOWER_BETTER."))
exports.save_table(KPE.round(4), "k_persistence", caption=(
    "Persona-paired K=0 − K=5 on the per-conversation persistence metrics (ct_persist, ct_relapse, st_to_ct), per "
    "grader, iterations 1..10, Holm across iterations within (judge, metric); better read with ct_relapse lower-is-better."))
display(KPE[KPE.p_holm < 0.05][["judge", "metric", "iteration", "mean_K0", "mean_K5", "dz", "p_holm", "better"]].round(3))

# persistence figure: pooled P(change talk | previous change talk) and P(change talk | previous sustain talk)
fig, axes = plt.subplots(1, len(PJ), figsize=(5.4 * len(PJ), 3.6), sharey=True)
for ax, j in zip(np.atleast_1d(axes), PJ):
    pe = PROC[j]["persistence"]
    for K in (0, 5):
        arm, st = f"GRPO_LA{K}", K_STYLE[K]
        for prev, alpha in (("CT", 1.0), ("ST", 0.55)):
            d = pe[(pe.arm == arm) & (pe.prev_code == prev)].sort_values("iteration")
            ax.plot(d.iteration, d.p_ct, ls=st["ls"], marker=st["marker"], ms=3.2, lw=1.3, color=PAL[arm], alpha=alpha,
                    label=f"{arm_label(arm)} · after {prev}")
    ax.set_title(j, fontsize=9); ax.set_xlabel("iteration (0 = Base)"); ax.grid(True, alpha=0.3); ax.set_ylim(0, 1)
np.atleast_1d(axes)[0].set_ylabel("P(next patient turn = change talk)")
np.atleast_1d(axes)[0].legend(fontsize=7, frameon=False, loc="center right")
fig.tight_layout()
exports.save_fig(fig, "persistence", caption=(
    "Change-talk persistence by iteration, pooled over turns (persistence_<judge>): P(the patient's next turn is "
    "change talk | the previous patient turn was change talk) — solid colour — and | sustain talk — faded; K=0 "
    "solid line, K=5 dashed; iteration 0 = the shared Base."))
plt.show()

## 3 · Sentence-embedding space on the shared Base

In [ ]:
UTT = text.load_utterances(KA).reset_index(drop=True)
E = text.embed_utterances(UTT)
TX = shared_base.text_tables(UTT, E, n_boot=300, seed=BOOT_SEED)
exports.save_table(TX["drift"].round(4), "text_drift", caption=(
    "Per (run, iteration): drift_norm = ‖therapist-turn centroid − shared Base centroid‖ (MiniLM, unit vectors), "
    "step_norm, cos_to_final. The Base centroid is the two base draws' union."))
exports.save_table(TX["drift_cosines"].round(4), "text_drift_cosines", caption=(
    "Per iteration: cosine between the two GRPO runs' displacement vectors (centroid − shared Base centroid). 1 = "
    "they moved the same way; 0 = orthogonal."))
exports.save_table(TX["diversity"].round(4), "text_diversity", caption=(
    "Per (run, iteration): template_sim (mean pairwise cosine across conversations at the same therapist turn "
    "index 1..8), persona_var_share (between-conversation / total embedding variance, 95% bootstrap over "
    "conversations), dup_rate, distinct-n, mean_words; the shared Base uses all 192 conversations."))
display(TX["drift_cosines"].round(3)); display(TX["diversity"][["arm", "iteration", "template_sim", "persona_var_share"]].round(3))

## 4 · The judge-free over-praise marker, session length, turn length

In [ ]:
TM = behavior.text_metrics(KA, attach_persona=True)
MK = shared_base.marker_and_length(TM)
exports.save_table(MK.round(4), "marker_and_length", caption=(
    "Per (run, iteration) on the shared Base: lex_overpraise_marker_rate (share of therapist turns with at least "
    "one of the ten over-praise patterns, mean over conversations — judge-free), conv_len (utterances), "
    "mean_turn_len (characters per therapist turn), n_th_turns, with SEs over conversations."))
display(MK.round(3))

## 5 · Saturation: SD by iteration, agreement per state, sign preservation (21 states)

In [ ]:
SD = shared_base.sd_tables(SB)
TR = shared_base.sd_trend(SD)
exports.save_table(SD.round(4), "sd_by_iter", caption=(
    "Per (judge, metric, run, iteration): n, mean, median, sd, iqr and the ceiling shares share_ge4 / share_ge45 / "
    "share_eq5 (replication.sd_by_iter) on the shared Base (192 conversations at iteration 0)."))
exports.save_table(TR.round(4), "sd_trend", caption=(
    "Per (judge, metric, run): Spearman ρ of the per-conversation SD against iteration (0..10 and 1..10) and the "
    "variance ratio of iteration 10 against the Base and against iteration 1."))
AGR = shared_base.agreement_by_state(ONE) if HELDOUT else pd.DataFrame()
ASUM = shared_base.agreement_summary(AGR) if not AGR.empty else pd.DataFrame()
PAIRS = shared_base.state_pair_contrasts(ONE) if HELDOUT else pd.DataFrame()
SIGN = reliability.sign_preservation(PAIRS)
OFF = shared_base.judge_offset(shared_base.levels(ONE)) if HELDOUT else pd.DataFrame()
exports.save_table(AGR.round(4), "agreement_by_state", caption=(
    "Per (state, instrument): per-conversation Pearson r between the two graders over the 21 model states "
    "(ONE Base: all 192 base conversations paired)."))
exports.save_table(ASUM.round(4), "agreement_summary", caption=(
    "Per instrument: r at the K=5 run's iteration 10, the median r over the 21 states, the difference, and that "
    "state's rank (1 = the instrument's worst-agreeing state). Compare down a column, never across instruments."))
exports.save_table(PAIRS.round(4), "state_pairs", caption=(
    "Every pair of the 21 states × instrument under both graders, persona-paired (the Base's persona value is the "
    "mean of its two conversations): deltas, dz, the held-out side's bootstrap CI (BOOT_SEED) and same_sign."))
exports.save_table(SIGN, "sign_preservation", caption=(
    "How often the held-out grader keeps the sign of the training oracle's state contrast, by the size of the "
    "contrast (reliability.sign_preservation over state_pairs: 8 instruments × C(21,2) pairs)."))
exports.save_table(OFF.round(4), "judge_offset", caption=(
    "Per state: the training oracle's mean Q1+Q2 minus the held-out grader's (21 states)."))
display(TR.round(3)); display(ASUM.round(3)); display(SIGN)

In [ ]:
NUM = shared_base.shared_base_numbers(lv=LV, gains_t=GAINS, sig=SIG, proc=PROC, text_t=TX, marker=MK, sd=SD, trend=TR,
                              agr=AGR, agr_sum=ASUM, sign=SIGN, offset=OFF)
exports.save_numbers("shared_base_numbers", NUM, caption=(
    "Ledger of the family's quotable scalars on the shared Base (both graders), each citing its table."))
print(f"{len(NUM)} ledger keys  [{time.time() - _T0:.0f} s]")

In [ ]:
exports.prune_orphan_captions(); exports.build_index()